In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


date_start = '2026-03-01'
date_end = '2026-04-30'

query_subscr = f'''
select
    contact_id,
    min(subscr_date_act::date) as first_subscribe_day
from subscr_status
group by contact_id
'''

subscr = su.execute_custom_query_gp(query_subscr)

subscr['first_subscribe_day'] = pd.to_datetime(
    subscr['first_subscribe_day']
)


scores = pd.concat(
    [
        march_scores.assign(score_month='2026-03'),
        april_scores.assign(score_month='2026-04')
    ],
    ignore_index=True
)

scores['score'] = pd.to_numeric(scores['score'], errors='coerce')

df = (
    scores
    .merge(subscr, on='contact_id', how='left')
)

df['is_subscribed'] = (
    df['first_subscribe_day'].between(
        pd.to_datetime(date_start),
        pd.to_datetime(date_end)
    )
).astype(int)


df['score_bin'] = pd.qcut(
    df['score'],
    q=10,
    duplicates='drop'
)

score_stats = (
    df
    .groupby(['score_month', 'score_bin'], as_index=False)
    .agg(
        clients_cnt=('contact_id', 'nunique'),
        subscribed_cnt=('is_subscribed', 'sum'),
        avg_score=('score', 'mean')
    )
)

score_stats['subscribed_pct'] = (
    score_stats['subscribed_cnt']
    / score_stats['clients_cnt']
    * 100
).round(2)

score_stats